# Data Pipeline — The Naylor Blueprint

Runs the full data pipeline:
1. **Discover runners** — sprint speed, hp_to_1b, SB/CS per season from Statcast
2. **Fetch leads** — per-attempt ground-covered data per qualified runner (cached)
3. **Ground covered leaderboard** — aggregate, residuals, rank, export to xlsx

**Network required** (Savant API). Cache in `Computer Vision/discovery/leads_cache/` — re-runs are instant.

---

In [ ]:
# Setup
import sys, subprocess
from pathlib import Path

REPO = Path().resolve()
# walk up until we find the repo root
for p in [REPO] + list(REPO.parents):
    if (p / 'Computer Vision' / 'Statcast Analysis Core').exists():
        REPO = p
        break

CORE = REPO / 'Computer Vision' / 'Statcast Analysis Core'
DISCOVERY = REPO / 'Computer Vision' / 'discovery'
DATA = REPO / 'Data Frame'

sys.path.insert(0, str(CORE))
print(f'Repo root : {REPO}')
print(f'Core      : {CORE}')
print(f'Discovery : {DISCOVERY}')

## 1. Discover Runners

Fetch sprint speed, home-to-1B time, and SB/CS for all seasons.

In [ ]:
SEASONS = [2023, 2024, 2025, 2026]

for year in SEASONS:
    out = DISCOVERY / f'runners_{year}_{year}.csv'
    if out.exists():
        import pandas as pd
        df = pd.read_csv(out)
        print(f'[{year}] cached — {len(df)} runners')
        continue
    print(f'[{year}] fetching...')
    r = subprocess.run(
        [sys.executable, str(CORE / 'discover_runners.py'),
         '--start', str(year), '--end', str(year), '--min-attempts', '1'],
        capture_output=True, text=True
    )
    print(r.stdout[-500:] if len(r.stdout) > 500 else r.stdout)
    if r.returncode != 0:
        print('ERROR:', r.stderr[:300])

In [ ]:
import pandas as pd

# Verify
for year in SEASONS:
    p = DISCOVERY / f'runners_{year}_{year}.csv'
    if p.exists():
        df = pd.read_csv(p)
        sample = df.iloc[0]
        print(f'{year}: {len(df)} runners | top: {sample["name"]} ({sample["sprint_speed_ftps"]:.1f} ft/s, {sample["sprint_pctile"]:.0f}th pctile)')

## 2. Fetch Per-Attempt Leads (Ground-Covered Data)

Pulls `gain_to_release_ft` per attempt from Savant for each runner.
Results cached to `Computer Vision/discovery/leads_cache/<runner_id>_<year>.csv`.

**First full run: ~15–20 min** (~700 API calls × 0.3s sleep). Re-runs: instant from cache.

In [ ]:
# Quick test — Naylor 2025
from fetch_leads import fetch_leads
import statistics

rows = fetch_leads(647304, 2025)  # Josh Naylor
gains = [r['gain_to_release_ft'] for r in rows if r.get('gain_to_release_ft') is not None]
print(f'Naylor 2025: {len(rows)} tracked attempts, mean gain = {statistics.mean(gains):.2f} ft')

## 3. Ground Covered Leaderboard

Aggregates all cached leads, computes speed-adjusted residual (gain beyond what sprint speed predicts),
ranks all qualified runner-seasons, and writes the result to `Data Frame/Naylor Blueprint.xlsx`
(sheet: **Ground Covered**).

In [ ]:
# Run the full leaderboard
# --min-attempts 5 for full seasons; 2026 handled internally with threshold=1
r = subprocess.run(
    [sys.executable, str(CORE / 'ground_covered_leaderboard.py'),
     '--min-attempts', '5', '--min-tracked', '1', '--sleep', '0.3'],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr[:500])

In [ ]:
# Verify output
xlsx = DATA / 'Naylor Blueprint.xlsx'
gc = pd.read_excel(xlsx, sheet_name='Ground Covered')
by_season = gc.groupby('season').agg(n=('player_name','count'), vq=('volume_qualified','sum')).reset_index()
print('Ground Covered sheet summary:')
print(by_season.to_string(index=False))

# Naylor/Soto check
ns = gc[gc['runner_id'].isin([647304, 665742])][
    ['player_name','season','sprint_pctile','n_tracked','mean_gain_to_release_ft','gain_residual_ft','rank_residual']
]
print('\nNaylor & Soto:')
print(ns.to_string(index=False))